# 7.3 Checklist change by energy relevance

Did the checklist actions that changed between BL and EL tend to be ones that can move our energy measure (the equipment-based kWh calculation)?

- Energy relevance of each action is the hand classification in `checklist_energy_classification.csv` (same as 7.2): "direct" (changes an input of the energy calculation), "indirect" (only via a weak/uncertain channel), "none". For the table, direct + indirect = energy-relevant.
- Unit: lab x action pairs where the lab gave a substantive answer ("Yes"/"No"/"I don't know") at *both* BL and EL, so a change is a lab switching answer, not moving in/out of N/A.
- Main outcome: switch to "Yes" among pairs not "Yes" at BL (i.e. that could still switch). Difference tested with an LPM, SEs clustered by lab.
- Then: for switches to "Yes" on *direct* actions, did the matching equipment input change in that lab's equipment data?

Only treated labs did the checklist, so these are BL-EL changes, not treatment effects. Both the checklist and the EL equipment survey were prefilled with BL answers.

Disclosure control: only pooled counts go in the table, and every count shown must be 0 or >= 5. Per-action breakdowns printed here are for internal use only.

Sample: treated labs with *both* BL and EL data (same as 7.1, 7.2).

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
import statsmodels.formula.api as smf

In [2]:
# Load data (blank cells are genuinely missing, not the string "NA")
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False,
    na_values=[""]
)

equipment = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_5.csv",
    keep_default_na=False,
    na_values=[""]
)

# Energy relevance of each action - edit the CSV to change the classification
energy_class = pd.read_csv(
    "checklist_energy_classification.csv", keep_default_na=False, na_values=[""]
).set_index("item")

## (1) Prepare data

In [3]:
# Treated labs with both BL and EL data - same sample as 7.1
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
matched_labgroupids = labgroup_counts[labgroup_counts == 2].index

df_bl = df[df["labgroupid"].isin(matched_labgroupids) & (df["survey"] == "BL")].copy()
df_treatment = df_bl[df_bl["treated"] == 1].copy()

print(f"Treated labs with both BL and EL data: {len(df_treatment)}")

Treated labs with both BL and EL data: 56


In [4]:
items = (
    [f"bronze_q_{i}" for i in range(1, 17)]
    + [f"silver_q_{i}" for i in range(1, 19)]
    + [f"gold_q_{i}" for i in range(1, 16)]
)

assert set(energy_class.index) == set(items), "CSV must have exactly one row per checklist item"
assert energy_class["energy_relevance"].isin(["direct", "indirect", "none"]).all()

In [5]:
# Long data: one row per lab x action, kept only if substantive at both BL and EL
substantive = ("Yes", "No", "I don't know")

rows = []
for item in items:
    resp_bl = df_treatment[f"{item}_bl"]
    resp_el = df_treatment[f"{item}_el"]
    is_substantive = resp_bl.isin(substantive) & resp_el.isin(substantive)
    rows.append(pd.DataFrame({
        "labgroupid": df_treatment.loc[is_substantive, "labgroupid"],
        "item": item,
        "bl_yes": resp_bl[is_substantive] == "Yes",
        "el_yes": resp_el[is_substantive] == "Yes",
    }))
pairs = pd.concat(rows, ignore_index=True)

pairs["relevance"] = pairs["item"].map(energy_class["energy_relevance"])
pairs["energy_relevant"] = (pairs["relevance"] != "none").astype(int)
pairs["to_yes"] = (~pairs["bl_yes"] & pairs["el_yes"]).astype(int)
pairs["from_yes"] = (pairs["bl_yes"] & ~pairs["el_yes"]).astype(int)

print(f"Lab x action pairs: {len(pairs)}, labs: {pairs['labgroupid'].nunique()}")

Lab x action pairs: 1446, labs: 48


## (2) Switch rates by energy relevance

In [6]:
def summarise(g):
    at_risk = ~g["bl_yes"]
    return pd.Series({
        "actions": g["item"].nunique(),
        "pairs": len(g),
        "bl_yes_pct": g["bl_yes"].mean() * 100,
        "at_risk": at_risk.sum(),
        "to_yes": g["to_yes"].sum(),
        "to_yes_pct": g.loc[at_risk, "to_yes"].mean() * 100,
        "from_yes": g["from_yes"].sum(),
        "net_change_pp": (g["el_yes"].mean() - g["bl_yes"].mean()) * 100,
    })

# Internal only - direct/indirect separately has cells < 5 (e.g. switches away from "Yes")
# print(pairs.groupby("relevance").apply(summarise, include_groups=False).round(1).T)

summary = pairs.groupby("energy_relevant").apply(summarise, include_groups=False)
# print(summary.round(1).T)

In [7]:
# LPMs, SEs clustered by lab
# (a) switch to "Yes" among pairs not "Yes" at BL
at_risk = pairs[~pairs["bl_yes"]]
reg_to_yes = smf.ols("to_yes ~ energy_relevant", data=at_risk).fit(
    cov_type="cluster", cov_kwds={"groups": at_risk["labgroupid"]}
)
# (c) "Yes" at BL, all pairs
pairs["bl_yes_int"] = pairs["bl_yes"].astype(int)
reg_bl = smf.ols("bl_yes_int ~ energy_relevant", data=pairs).fit(
    cov_type="cluster", cov_kwds={"groups": pairs["labgroupid"]}
)
print(f"Yes at BL: diff = {reg_bl.params['energy_relevant']:.3f} "
      f"(SE {reg_bl.bse['energy_relevant']:.3f}, p = {reg_bl.pvalues['energy_relevant']:.3f}), "
      f"n = {int(reg_bl.nobs)}, clusters = {pairs['labgroupid'].nunique()}")

# (b) net change in "Yes" (EL - BL), all pairs
pairs["dy"] = pairs["el_yes"].astype(int) - pairs["bl_yes"].astype(int)
reg_net = smf.ols("dy ~ energy_relevant", data=pairs).fit(
    cov_type="cluster", cov_kwds={"groups": pairs["labgroupid"]}
)

for name, reg in [("Switch to Yes", reg_to_yes), ("Net change", reg_net)]:
    print(f"{name}: diff = {reg.params['energy_relevant']:.3f} "
          f"(SE {reg.bse['energy_relevant']:.3f}, p = {reg.pvalues['energy_relevant']:.3f}), "
          f"n = {int(reg.nobs)}, clusters = {pairs['labgroupid'].nunique() if name == 'Net change' else at_risk['labgroupid'].nunique()}")

# Direct and indirect separately (vs none) - for the text
reg_3 = smf.ols("to_yes ~ C(relevance, Treatment('none'))", data=at_risk).fit(
    cov_type="cluster", cov_kwds={"groups": at_risk["labgroupid"]}
)
print(reg_3.summary2().tables[1][["Coef.", "Std.Err.", "P>|z|"]].round(3))

Yes at BL: diff = 0.004 (SE 0.022, p = 0.849), n = 1446, clusters = 48
Switch to Yes: diff = 0.056 (SE 0.044, p = 0.207), n = 372, clusters = 47
Net change: diff = 0.009 (SE 0.013, p = 0.469), n = 1446, clusters = 48


                                             Coef.  Std.Err.  P>|z|
Intercept                                    0.135     0.025  0.000
C(relevance, Treatment('none'))[T.direct]    0.043     0.067  0.524
C(relevance, Treatment('none'))[T.indirect]  0.065     0.052  0.210


## (3) Do switches to "Yes" on direct actions show up in the equipment data?

Each direct action is matched to the equipment inputs it acts on. For each lab x action pair, "input changed" = any of that lab's matching equipment items (same equipment, type_no at BL and EL) has a different value of any of those inputs at EL.

In [8]:
# Direct action -> (equipment type, inputs); None = any equipment
direct_inputs = {
    "bronze_q_1": (None, ["hours", "days"]),
    "bronze_q_10": ("ult", ["seals", "spacing", "filter", "door_openings"]),
    "bronze_q_16": ("fc", ["hours_open"]),
    "silver_q_18": ("fc", ["lifted"]),
    "gold_q_15": ("fc", ["face_velocity"]),
    "silver_q_17": ("bath", ["lid", "hours"]),
}
# Keep in sync with the CSV
assert set(direct_inputs) == set(energy_class.index[energy_class["energy_relevance"] == "direct"])

# Equipment items observed at both BL and EL
key = ["labgroupid", "equipment", "type_no"]
eq_bl = equipment[equipment["survey"] == "BL"].set_index(key)
eq_el = equipment[equipment["survey"] == "EL"].set_index(key)
matched = eq_bl.index.intersection(eq_el.index)
eq_bl, eq_el = eq_bl.loc[matched], eq_el.loc[matched]

def input_changed(col):
    both_missing = eq_bl[col].isna() & eq_el[col].isna()
    return (eq_bl[col] != eq_el[col]) & ~both_missing

link_rows = []
for item, (eq_type, cols) in direct_inputs.items():
    changed = pd.concat([input_changed(c) for c in cols], axis=1).any(axis=1)
    if eq_type is not None:
        changed = changed[changed.index.get_level_values("equipment") == eq_type]
    lab_changed = changed.groupby(level="labgroupid").any()

    p = pairs[pairs["item"] == item].copy()
    p["group"] = np.select(
        [p["to_yes"] == 1, ~p["bl_yes"] & ~p["el_yes"], p["bl_yes"] & p["el_yes"]],
        ["switched to Yes", "not Yes at both", "Yes at both"],
        default="switched away from Yes",
    )
    p = p[p["labgroupid"].isin(lab_changed.index)]  # labs with matching equipment
    p["input_changed"] = p["labgroupid"].map(lab_changed).astype(int)
    link_rows.append(p)

link = pd.concat(link_rows)

# Internal only - per-action cells are < 5
# print(link.groupby(["item", "group"])["input_changed"].agg(["size", "sum"]))

link_pooled = link.groupby("group")["input_changed"].agg(pairs="size", input_changed="sum")
#print(link_pooled)

## (4) Build and save table

In [ ]:
def fmt(v, decimals=1):
    s = f"{abs(v):.{decimals}f}"
    return f"$\\llap{{-}}{s}$" if v < 0 else f"${s}$"

def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""

def fmt_diff(reg):
    b, p = reg.params["energy_relevant"] * 100, reg.pvalues["energy_relevant"]
    s = f"{abs(b):.1f}"
    return f"$\\llap{{-}}{s}\\rlap{{{stars(p)}}}$" if b < 0 else f"${s}\\rlap{{{stars(p)}}}$"

def fmt_se(reg):
    return f"(${reg.bse['energy_relevant'] * 100:.1f}$)"

all_pairs, none, energy = summarise(pairs), summary.loc[0], summary.loc[1]

def counts(col):
    return f"${int(all_pairs[col])}$ & ${int(none[col])}$ & ${int(energy[col])}$"

def shares(col):
    return f"{fmt(all_pairs[col])} & {fmt(none[col])} & {fmt(energy[col])}"

# Disclosure check: every count in the table must be 0 or >= 5
table_counts = [g[col] for g in (all_pairs, none, energy)
                for col in ("actions", "pairs", "at_risk", "to_yes")]
assert all(c == 0 or c >= 5 for c in table_counts), "A count in the table is below 5"

lines = [
    r"\begin{tabular}{@{}L{9cm}C{2cm}C{2cm}C{2cm}C{1.8cm}}",
    r"\hline",
    r"\addlinespace[0.2cm]",
    r" & All & Not energy-relevant & Energy-relevant & \\",
    r" & (1) & (2) & (3) & (3)-(2) \\",
    r"\hline",
    r"\addlinespace[0.2cm]",
    f"Share ``Yes'' at BL (\\%) & {shares('bl_yes_pct')} & {fmt_diff(reg_bl)} \\\\",
    f" & & & & {fmt_se(reg_bl)} \\\\",
    r"\addlinespace[0.2cm]",
    f"Switched to ``Yes'' (\\% of pairs not ``Yes'' at BL) & {shares('to_yes_pct')} & {fmt_diff(reg_to_yes)} \\\\",
    f" & & & & {fmt_se(reg_to_yes)} \\\\",
    r"\addlinespace[0.2cm]",
    r"\hline",
    r"\addlinespace[0.2cm]",
    f"Checklist actions & {counts('actions')} & \\\\",
    f"Group--action pairs & {counts('pairs')} & \\\\",
    f"Group--action pairs not ``Yes'' at BL & {counts('at_risk')} & \\\\",
    f"Switched to ``Yes'' at EL & {counts('to_yes')} & \\\\",
    r"\addlinespace[0.2cm]",
    r"\hline",
    r"\end{tabular}",
]
table = "\n".join(lines)
# print(table)

out_dir = config.OUTPUT / "9_Checklist_Tables"
out_dir.mkdir(parents=True, exist_ok=True)
table_path = out_dir / "checklist_energy_change_table.tex"
_ = table_path.write_text(table)

\begin{tabular}{@{}L{9cm}C{2cm}C{2cm}C{2cm}C{1.8cm}}
\hline
\addlinespace[0.2cm]
 & All & Not energy-relevant & Energy-relevant & \\
 & (1) & (2) & (3) & (3)-(2) \\
\hline
\addlinespace[0.2cm]
Share ``Yes'' at BL (\%) & $74.3$ & $74.2$ & $74.6$ & $0.4\rlap{}$ \\
 & & & & ($2.2$) \\
\addlinespace[0.2cm]
Switched to ``Yes'' (\% of pairs not ``Yes'' at BL) & $15.1$ & $13.5$ & $19.0$ & $5.6\rlap{}$ \\
 & & & & ($4.4$) \\
\addlinespace[0.2cm]
\hline
\addlinespace[0.2cm]
Checklist actions & $49$ & $35$ & $14$ & \\
Group--action pairs & $1446$ & $1033$ & $413$ & \\
Group--action pairs not ``Yes'' at BL & $372$ & $267$ & $105$ & \\
Switched to ``Yes'' at EL & $56$ & $36$ & $20$ & \\
\addlinespace[0.2cm]
\hline
\end{tabular}
